# Phase 4: Feature Normalization and Stronger Ranking Baselines

Objective: To evaluate whether feature normalization improves ranking robustness and early-rank performance.


### Phase 4 Goals:

Test whether normalization affects:
1. Aggregate metrics (P@K, NDCG@K, Failure@K)
2. Conditional failure rates (model-focused on feasible queries)
3. Avoidable failures (queries where relevant docs exist)
4. Score flatness and ties (error taxonomy component)
5. Generalization to MQ2008 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ndcg_score  
import pickle
import warnings
warnings.filterwarnings('ignore')

#Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

#Setup paths
PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED=PROJECT_ROOT / 'data' / 'processed'
PHASE3_OUTPUT=DATA_PROCESSED / 'phase3_baseline'
PHASE4_OUTPUT=DATA_PROCESSED / 'phase4_normalization'
PHASE4_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Phase 4 output directory: {PHASE4_OUTPUT}")

#Constants
K_VALUES=[1, 3, 5, 10]
ZERO_VAR_FEATURES=[f'f{i}' for i in range(6, 11)]  # f6-f10

Project root: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding
Phase 4 output directory: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase4_normalization


# 1. Data Loading

In [5]:
def parse_letor_file(filepath):
    """Parsing LETOR format file."""
    data=[]
    with open(filepath, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line=line.strip()
            if not line:
                continue
            try:
                parts=line.split('#')
                feature_part=parts[0].strip()
                docid=None
                if len(parts)>1:
                    comment=parts[1].strip()
                    if 'docid' in comment:
                        docid=comment.split('=')[1].strip()
                tokens=feature_part.split()
                label=int(tokens[0])
                qid=int(tokens[1].split(':')[1])
                features={}
                for token in tokens[2:]:
                    fid, value=token.split(':')
                    features[int(fid)]=float(value)
                data.append({
                    'label': label,
                    'qid': qid,
                    'features': features,
                    'docid': docid
                })
            except Exception as e:
                print(f"Error parsing line {line_num}: {e}")
                continue
    return data


def data_to_dataframe(data):
    """Converting parsed data to DataFrame."""
    records=[]
    for item in data:
        record={'label': item['label'], 'qid': item['qid'], 'docid': item['docid']}
        for fid, value in item['features'].items():
            record[f'f{fid}']=value
        records.append(record)
    return pd.DataFrame(records)


#Loading MQ2007 Fold1
print("Loading MQ2007 Fold1...")
train_df_2007=pd.read_csv(DATA_PROCESSED / 'fold1_train_processed.csv')

FOLD1_DIR_2007=DATA_RAW / 'MQ2007' / 'Fold1'
test_file_2007=FOLD1_DIR_2007 / 'test.txt'

test_data_2007=parse_letor_file(str(test_file_2007))
test_df_2007=data_to_dataframe(test_data_2007)

print(f"  MQ2007 Train: {train_df_2007.shape} ({train_df_2007['qid'].nunique()} queries)")
print(f"  MQ2007 Test: {test_df_2007.shape} ({test_df_2007['qid'].nunique()} queries)")
print(f"  Evaluating on: TEST SPLIT")

#Loading MQ2008 Fold1
print("\nLoading MQ2008 Fold1...")
FOLD1_DIR_2008=DATA_RAW / 'MQ2008' / 'Fold1'

if not FOLD1_DIR_2008.exists():
    raise FileNotFoundError(
        f"MQ2008 directory not found at: {FOLD1_DIR_2008}\n"
        f"Please download MQ2008 and extract to: {DATA_RAW / 'MQ2008'}\n"
        "Expected structure: data/raw/MQ2008/Fold1/train.txt and test.txt"
    )

train_file_2008=FOLD1_DIR_2008 / 'train.txt'
test_file_2008=FOLD1_DIR_2008 / 'test.txt'


if not (train_file_2008.exists() and test_file_2008.exists()):
    raise FileNotFoundError(f"MQ2008 train/test files not found in {FOLD1_DIR_2008}")

train_data_2008=parse_letor_file(str(train_file_2008))
test_data_2008=parse_letor_file(str(test_file_2008))

train_df_2008=data_to_dataframe(train_data_2008)
test_df_2008=data_to_dataframe(test_data_2008)

print(f"  MQ2008 Train: {train_df_2008.shape} ({train_df_2008['qid'].nunique()} queries)")
print(f"  MQ2008 Test: {test_df_2008.shape} ({test_df_2008['qid'].nunique()} queries)")

#Identifying features
feature_cols=[col for col in train_df_2007.columns if col.startswith('f') and col[1:].isdigit()]
feature_cols=sorted(feature_cols, key=lambda x: int(x[1:]))
active_features=[f for f in feature_cols if f not in ZERO_VAR_FEATURES]

print(f"\nFeatures:")
print(f"  Total: {len(feature_cols)}")
print(f"  Zero-variance excluded: {ZERO_VAR_FEATURES}")
print(f"  Active: {len(active_features)}")


#Ensuring all feature columns exist (f1..f46)
ALL_FEATURES=[f"f{i}" for i in range(1, 47)]

for df_name, df in [("test_df_2007", test_df_2007),
                    ("train_df_2008", train_df_2008),
                    ("test_df_2008", test_df_2008)]:
    missing=[c for c in ALL_FEATURES if c not in df.columns]
    if missing:
        for c in missing:
            df[c]=0.0
        print(f"Filled {len(missing)} missing feature columns in {df_name}: {missing[:5]}{'...' if len(missing)>5 else ''}")

#ensuring consistent column order
test_df_2007 = test_df_2007[['label','qid','docid'] + ALL_FEATURES]
train_df_2008 = train_df_2008[['label','qid','docid'] + ALL_FEATURES]
test_df_2008  = test_df_2008[['label','qid','docid'] + ALL_FEATURES]


Loading MQ2007 Fold1...
  MQ2007 Train: (42158, 49) (1017 queries)
  MQ2007 Test: (13652, 49) (336 queries)
  Evaluating on: TEST SPLIT

Loading MQ2008 Fold1...
  MQ2008 Train: (9630, 49) (471 queries)
  MQ2008 Test: (2874, 49) (156 queries)

Features:
  Total: 46
  Zero-variance excluded: ['f6', 'f7', 'f8', 'f9', 'f10']
  Active: 41


## 1.1 Verifying consistency with Phase 3

In [6]:
#Loading Phase 3 results for verification
phase3_query_metrics=pd.read_csv(PHASE3_OUTPUT / 'baseline_query_metrics.csv')

print("Phase 3 Baseline (for verification):")
print(f"  Test queries: {len(phase3_query_metrics)}")
print(f"  Failure@5 (primary): {phase3_query_metrics['Failure@5_primary'].mean():.4f}")
print(f"  Queries with relevant docs: {(phase3_query_metrics['num_relevant_1'] > 0).sum()}")

queries_with_rel=phase3_query_metrics[phase3_query_metrics['num_relevant_1'] > 0]
print(f"  Conditional Failure@5: {queries_with_rel['Failure@5_primary'].mean():.4f}")

print(f"\nPhase 4 will evaluate on the same test split to ensure comparability.")

Phase 3 Baseline (for verification):
  Test queries: 336
  Failure@5 (primary): 0.2679
  Queries with relevant docs: 290
  Conditional Failure@5: 0.1517

Phase 4 will evaluate on the same test split to ensure comparability.


# 2. Evaluation Functions (Phase 3 Validated)

In [ ]:
def precision_at_k(ranked_labels, k, relevance_threshold=1):
    """
    Computing Precision@K with validation.
    Uses denominator min(k, num_docs) as per Phase 3.
    """
    if relevance_threshold not in [1, 2]:
        raise ValueError(f"Invalid relevance_threshold: {relevance_threshold}")
    
    if k<=0:
        raise ValueError(f"Invalid k:{k}")
    
    if len(ranked_labels)==0:
        return 0.0
    
    k_actual=min(k, len(ranked_labels))
    top_k=ranked_labels[:k_actual]
    
    if relevance_threshold==1:
        relevant_count=sum(label>=1 for label in top_k)
    else:
        relevant_count=sum(label==2 for label in top_k)
    
    return relevant_count / k_actual


def is_failure_at_k(ranked_labels, k, relevance_threshold=1):
    """
    Determine if query is a failure at K with validation.
    """
    if relevance_threshold not in [1, 2]:
        raise ValueError(f"Invalid relevance_threshold: {relevance_threshold}")
    
    if k<=0:
        raise ValueError(f"Invalid k: {k}")
    
    if len(ranked_labels)==0:
        return True
    
    top_k_labels=ranked_labels[:k]
    
    if relevance_threshold==1:
        has_relevant=any(label>=1 for label in top_k_labels)
    else:
        has_relevant=any(label==2 for label in top_k_labels)
    
    return not has_relevant


def ndcg_at_k(ranked_labels, k):
    """
    Computing NDCG@K (gain-matched with sklearn).
    Verified in Phase 3.
    """
    if k<=0:
        raise ValueError(f"Invalid k:{k}")

    if len(ranked_labels)==0:
        return 0.0
    
    #Computing DCG@K
    dcg=0.0
    for i, label in enumerate(ranked_labels[:k]):
        dcg+=(2**label - 1) / np.log2(i + 2)
    
    #Computing IDCG@K
    ideal_labels=sorted(ranked_labels, reverse=True)[:k]
    idcg=0.0
    for i, label in enumerate(ideal_labels):
        idcg+=(2**label - 1) / np.log2(i + 2)
    
    if idcg==0:
        return 0.0
    
    return dcg / idcg


print("Evaluation functions defined (Phase 3 validated)")

Evaluation functions defined (Phase 3 validated)


In [ ]:
print("="*80)
print("NDCG VERIFICATION (Gain-Matched with sklearn)")
print("="*80)

#Sampling 5 queries from test set
sample_qids=test_df_2007['qid'].unique()[:5]
max_diff=0.0

print("\nVerifying NDCG computation on sample queries:")
for qid in sample_qids:
    query_data=test_df_2007[test_df_2007['qid']==qid]
    labels=query_data['label'].values
    if len(labels)==0:
        continue

    #Gain-matched: our ndcg uses (2^label-1)
    gains=(2**labels)-1

    y_score=np.arange(len(labels),0,-1)
    
    for k in [3, 5, 10]:
        #Our implementation
        our_ndcg=ndcg_at_k(labels, k)
        
        #sklearn implementation (reshape for API)
        if len(labels)>0:
            sklearn_ndcg=ndcg_score(
                [gains], 
                [y_score],  
                k=k
            )
            diff=abs(our_ndcg - sklearn_ndcg)
            max_diff=max(max_diff, diff)
            
            if diff>1e-9:
                print(f" qid={qid}, K={k}: ours={our_ndcg:.6f}, sklearn={sklearn_ndcg:.6f}, diff={diff:.2e}")

print(f"\nMaximum difference: {max_diff:.2e}")
if max_diff < 1e-9:
    print("NDCG implementation verified (gain-matched with sklearn)")
else:
    print(f"Warning: NDCG difference exceeds tolerance")

NDCG VERIFICATION (Gain-Matched with sklearn)

Verifying NDCG computation on sample queries:

Maximum difference: 8.33e-17
NDCG implementation verified (gain-matched with sklearn)


# 3. Normalization function

In [18]:
def normalize_global(train_features, test_features):    #computing statistics once on the training set, then applying them everywhere
    """
    Applying global StandardScaler normalization.
    
    Parameters:
    -----------
    train_features: array(n_train, n_features)
    test_features: array(n_test, n_features)
    
    Returns:
    --------
    train_norm, test_norm, scaler
    """
    scaler=StandardScaler()
    train_norm=scaler.fit_transform(train_features) #fit computes mean and std. transform applies the formula
    test_norm=scaler.transform(test_features)       #uses the mean and std from train_features to avoid leakage
    return train_norm, test_norm, scaler


#As phase 2 showed, there is a very high within-query variance
#So we are doing normalization per query below
def normalize_per_query(df, feature_cols, qid_col='qid'):
    """
    Applying per-query z-score normalization.
    Uses ddof=0 consistently (population std).
    Guards against zero-variance within query.
    
    Parameters:
    -----------
    df: DataFrame with qid and features
    feature_cols: list of feature column names
    
    Returns:
    --------
    df_normalized: DataFrame copy with normalized features
    """

    #Normalizes inside each query, independently
    #Each query gets its own mean and std
    df_norm=df.copy()
    
    for qid, group in df.groupby(qid_col):
        for feat in feature_cols:
            values=group[feat].values
            mean=values.mean()
            std=values.std(ddof=0)  
            #ddof=deltas degrees of freedom
            #Population std (ddof=0)
            #We are doing normalization here, so we use all the data we have
            #We don't assume there is hidden data to make ddof=1
            
            if std>1e-10:  #Guard against zero-variance
                normalized=(values-mean)/std
            else:
                normalized=values -mean  #Center only
            
            df_norm.loc[group.index, feat]=normalized
    
    return df_norm


print(" Normalization functions defined")
print("\nNormalization details:")
print("  Global: StandardScaler (z-score) fitted on train")
print("  Per-Query: Z-score within query, ddof=0, guards zero-variance")

 Normalization functions defined

Normalization details:
  Global: StandardScaler (z-score) fitted on train
  Per-Query: Z-score within query, ddof=0, guards zero-variance


# 4. Pipeline Runner (Clean API)

In [21]:
def run_pipeline(train_df, test_df, active_features, pipeline_name):
    """
    Running complete pipeline: normalize, train, predict, rank.
    
    Pipeline behavior controlled by pipeline_name only (clean API).
    
    Parameters:
    -----------
    train_df, test_df: DataFrames
    active_features: list of feature names
    pipeline_name: str ('raw', 'global', 'per_query')
    
    Returns:
    --------
    results : dict with model, test_results, pipeline_name
    """
    print(f"\n{'='*80}")
    print(f"PIPELINE: {pipeline_name.upper()}")
    print(f"{'='*80}")
    
    #Applying normalization based on pipeline_name
    if pipeline_name=='raw':
        X_train=train_df[active_features].values
        X_test=test_df[active_features].values
        print(" Normalization: None (raw features)")
        
    elif pipeline_name=='global':
        X_train, X_test, scaler=normalize_global(
            train_df[active_features].values,
            test_df[active_features].values
        )
        print(" Normalization: Global StandardScaler (z-score)")
        
    elif pipeline_name=='per_query':
        train_norm=normalize_per_query(train_df, active_features)
        test_norm=normalize_per_query(test_df, active_features)
        X_train=train_norm[active_features].values
        X_test=test_norm[active_features].values
        print("Normalization: Per-query z-score (ddof=0)")
        
    else:
        raise ValueError(f"Unknown pipeline: {pipeline_name}")
    
    y_train=train_df['label'].values
    
    print(f"Train shape: {X_train.shape}")
    print(f"Test shape: {X_test.shape}")
    
    #Training model
    print("\nTraining Logistic Regression...")
    model=LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=1000,
        random_state=36
    )
    model.fit(X_train, y_train)
    print("Model trained")
    
    #Predict scores (E[y])
    y_pred_proba=model.predict_proba(X_test)
    classes=model.classes_
    scores=np.sum(y_pred_proba*classes.reshape(1, -1), axis=1)
    #reshaping turns [0,1,2] into [[0,1,2]] because numpy broadcasting requires compatible shapes
    #y_pred_proba = [[0.7,0.2,0.1],[0.1,0.6,0.3]] for doc1 and doc2 respectively
    #Multiplying gets [[0.0,0.2,0.2],[0.0,0.6,0.6]]
    #axis=1 means sum across all classes

    #Creating results dataframe
    test_results=test_df.copy()
    test_results['score']=scores
    test_results['rank']=test_results.groupby('qid')['score'].rank(ascending=False, method='first').astype(int)
    
    print(f"\n  Score statistics:")
    print(f"    Min: {scores.min():.4f}")
    print(f"    Max: {scores.max():.4f}")
    print(f"    Mean: {scores.mean():.4f}")
    print(f"    Std: {scores.std():.4f}")
    
    return {
        'model': model,
        'test_results': test_results,
        'pipeline_name': pipeline_name
    }


print("Pipeline runner defined (clean API - controlled by pipeline_name only)")

Pipeline runner defined (clean API - controlled by pipeline_name only)


# 5. Evaluation function

In [24]:
def evaluate_pipeline(test_results, k_values=K_VALUES):
    """
    Computing comprehensive metrics including tie diagnostics.
    
    Returns:
    --------
    query_metrics: DataFrame(query-level)
    """
    query_results=[]
    
    for qid in test_results['qid'].unique():
        query_data=test_results[test_results['qid'] == qid].sort_values('rank')
        ranked_labels=query_data['label'].values
        ranked_scores=query_data['score'].values
        
        result={
            'qid': qid,
            'num_docs': len(ranked_labels),
            'num_relevant_1': sum(ranked_labels >= 1),
            'num_relevant_2': sum(ranked_labels == 2),
        }
        
        #Computing metrics for each K
        for k in k_values:
            #Primary (label >= 1)
            result[f'P@{k}_primary']=precision_at_k(ranked_labels, k, 1)
            result[f'NDCG@{k}']=ndcg_at_k(ranked_labels, k)
            result[f'Failure@{k}_primary']=is_failure_at_k(ranked_labels, k, 1)
            
            #Sensitivity (label == 2)
            result[f'P@{k}_sensitivity']=precision_at_k(ranked_labels, k, 2)
            result[f'Failure@{k}_sensitivity']=is_failure_at_k(ranked_labels, k, 2)
            
            #Tie diagnostics (only for K=5 and K=10)
            if k in [5, 10]:
                k_actual=min(k, len(ranked_scores))
                top_k_scores=ranked_scores[:k_actual]
                unique_scores=len(np.unique(top_k_scores))
                result[f'unique_scores@{k}']=unique_scores
                result[f'num_ties@{k}']=k_actual - unique_scores
                
                #Tie at cutoff
                if len(ranked_scores)>k:
                    result[f'tie_at_cutoff@{k}']=(ranked_scores[k-1]==ranked_scores[k])
                else:
                    result[f'tie_at_cutoff@{k}']=False
        
        #Score spread in top-10
        top_10_scores=ranked_scores[:min(10, len(ranked_scores))]
        if len(top_10_scores)>0:
            result['score_range_top10']=top_10_scores.max()-top_10_scores.min()
            result['score_std_top10']=top_10_scores.std(ddof=0)
        else:
            result['score_range_top10']=0.0
            result['score_std_top10']=0.0
        
        query_results.append(result)
    
    return pd.DataFrame(query_results)


print("Evaluation function defined")

Evaluation function defined


# 6. Running All Pipelines on MQ2007

In [26]:
print("="*80)
print("RUNNING ALL PIPELINES ON MQ2007 FOLD1")
print("="*80)

pipelines_2007={}
query_metrics_2007={}

for pipeline_name in ['raw', 'global', 'per_query']:
    #Running pipeline
    pipeline_result=run_pipeline(
        train_df_2007, test_df_2007, active_features, pipeline_name
    )
    pipelines_2007[pipeline_name]=pipeline_result
    
    #Evaluate
    print(f"Evaluating {pipeline_name}...")
    query_metrics=evaluate_pipeline(pipeline_result['test_results'])
    query_metrics_2007[pipeline_name]=query_metrics
    print(f"Evaluated {len(query_metrics)} queries\n")

print("="*80)
print("ALL MQ2007 PIPELINES COMPLETE")
print("="*80)

RUNNING ALL PIPELINES ON MQ2007 FOLD1

PIPELINE: RAW
 Normalization: None (raw features)
Train shape: (42158, 41)
Test shape: (13652, 41)

Training Logistic Regression...
Model trained

  Score statistics:
    Min: 0.0168
    Max: 1.1613
    Mean: 0.3104
    Std: 0.1693
Evaluating raw...
Evaluated 336 queries


PIPELINE: GLOBAL
 Normalization: Global StandardScaler (z-score)
Train shape: (42158, 41)
Test shape: (13652, 41)

Training Logistic Regression...
Model trained

  Score statistics:
    Min: 0.0108
    Max: 1.2909
    Mean: 0.3112
    Std: 0.1717
Evaluating global...
Evaluated 336 queries


PIPELINE: PER_QUERY
Normalization: Per-query z-score (ddof=0)
Train shape: (42158, 41)
Test shape: (13652, 41)

Training Logistic Regression...
Model trained

  Score statistics:
    Min: 0.0123
    Max: 1.3347
    Mean: 0.3021
    Std: 0.1288
Evaluating per_query...
Evaluated 336 queries

ALL MQ2007 PIPELINES COMPLETE


# 7. Running All Pipelines on MQ2008

In [28]:
print("="*80)
print("RUNNING ALL PIPELINES ON MQ2008 FOLD1")
print("="*80)

pipelines_2008={}
query_metrics_2008={}

for pipeline_name in ['raw', 'global', 'per_query']:
    #Run pipeline
    pipeline_result=run_pipeline(
        train_df_2008, test_df_2008, active_features, pipeline_name
    )
    pipelines_2008[pipeline_name]=pipeline_result
    
    #Evaluate
    print(f"Evaluating {pipeline_name}...")
    query_metrics=evaluate_pipeline(pipeline_result['test_results'])
    query_metrics_2008[pipeline_name]=query_metrics
    print(f"Evaluated {len(query_metrics)} queries\n")

print("=" * 80)
print("ALL MQ2008 PIPELINES COMPLETE")
print("=" * 80)

RUNNING ALL PIPELINES ON MQ2008 FOLD1

PIPELINE: RAW
 Normalization: None (raw features)
Train shape: (9630, 41)
Test shape: (2874, 41)

Training Logistic Regression...
Model trained

  Score statistics:
    Min: 0.0083
    Max: 1.2449
    Mean: 0.2733
    Std: 0.2395
Evaluating raw...
Evaluated 156 queries


PIPELINE: GLOBAL
 Normalization: Global StandardScaler (z-score)
Train shape: (9630, 41)
Test shape: (2874, 41)

Training Logistic Regression...
Model trained

  Score statistics:
    Min: 0.0064
    Max: 1.2594
    Mean: 0.2718
    Std: 0.2416
Evaluating global...
Evaluated 156 queries


PIPELINE: PER_QUERY
Normalization: Per-query z-score (ddof=0)
Train shape: (9630, 41)
Test shape: (2874, 41)

Training Logistic Regression...
Model trained

  Score statistics:
    Min: 0.0053
    Max: 1.4178
    Mean: 0.2525
    Std: 0.1955
Evaluating per_query...
Evaluated 156 queries

ALL MQ2008 PIPELINES COMPLETE


Looking at the std of both MQ2007 and MQ2008, the model's E[y] predictions are more spread out overall for 2008 data. So, MQ2008 might be easier for the model to separate docs (stronger signal) OR the class distribution differs, causing different calibration.

# 8. Computing Aggregate Metrics (Separate Primary/Sensitivity)

In [30]:
def compute_aggregate_metrics(query_metrics_dict, dataset_name):
    """
    Computing aggregate metrics separately for primary and sensitivity.
    
    Returns:
    --------
    agg_primary, agg_sensitivity: DataFrames
    """
    primary_rows=[]
    sensitivity_rows=[]
    
    for pipeline_name, qm in query_metrics_dict.items():
        for k in K_VALUES:
            #Primary metrics
            primary_row={
                'dataset': dataset_name,
                'pipeline': pipeline_name,
                'K': k,
                'P@K': qm[f'P@{k}_primary'].mean(),
                'NDCG@K': qm[f'NDCG@{k}'].mean(),
                'Failure@K': qm[f'Failure@{k}_primary'].mean(),
            }
            
            #Conditional failure (primary)
            queries_with_rel=qm[qm['num_relevant_1']>0]
            if len(queries_with_rel)>0:
                primary_row['Conditional_Failure@K']=queries_with_rel[f'Failure@{k}_primary'].mean()
                primary_row['num_queries_with_relevant']=len(queries_with_rel)
            else:
                primary_row['Conditional_Failure@K']=np.nan
                primary_row['num_queries_with_relevant']=0
            
            primary_rows.append(primary_row)
            
            #Sensitivity metrics
            sensitivity_row={
                'dataset': dataset_name,
                'pipeline': pipeline_name,
                'K': k,
                'P@K': qm[f'P@{k}_sensitivity'].mean(),
                'Failure@K': qm[f'Failure@{k}_sensitivity'].mean(),
            }
            
            #Conditional failure (sensitivity)
            queries_with_high_rel=qm[qm['num_relevant_2']>0]
            if len(queries_with_high_rel)> 0:
                sensitivity_row['Conditional_Failure@K']=queries_with_high_rel[f'Failure@{k}_sensitivity'].mean()
                sensitivity_row['num_queries_with_relevant']=len(queries_with_high_rel)
            else:
                sensitivity_row['Conditional_Failure@K']=np.nan
                sensitivity_row['num_queries_with_relevant']=0
            
            sensitivity_rows.append(sensitivity_row)
    
    return pd.DataFrame(primary_rows), pd.DataFrame(sensitivity_rows)


#Computing for MQ2007
agg_primary_2007, agg_sensitivity_2007=compute_aggregate_metrics(query_metrics_2007, 'MQ2007')

#Computing for MQ2008
agg_primary_2008, agg_sensitivity_2008=compute_aggregate_metrics(query_metrics_2008, 'MQ2008')

print("Aggregate metrics computed (primary and sensitivity separated)")

Aggregate metrics computed (primary and sensitivity separated)


In [31]:
#Phase 3 vs Phase 4 RAW replication check
phase4_raw = agg_primary_2007[
    (agg_primary_2007["pipeline"] == "raw") &
    (agg_primary_2007["K"] == 5)
]["Failure@K"].iloc[0]

print("Phase 3 Failure@5:", 0.2679)
print("Phase 4 RAW Failure@5:", phase4_raw)
print("Difference:", phase4_raw - 0.2679)


Phase 3 Failure@5: 0.2679
Phase 4 RAW Failure@5: 0.26785714285714285
Difference: -4.285714285717779e-05


## 8.1 Displaying MQ2007 Results

In [33]:
print("="*80)
print("MQ2007 AGGREGATE RESULTS")
print("="*80)

print("\nPRIMARY (label >= 1):")
display(agg_primary_2007[['pipeline', 'K', 'P@K', 'NDCG@K', 'Failure@K', 'Conditional_Failure@K']])

print("\nSENSITIVITY (label == 2):")
display(agg_sensitivity_2007[['pipeline', 'K', 'P@K', 'Failure@K', 'Conditional_Failure@K']])

MQ2007 AGGREGATE RESULTS

PRIMARY (label >= 1):


,pipeline,K,P@K,NDCG@K,Failure@K,Conditional_Failure@K
0,raw,1,0.4643,0.4127,0.5357,0.4621
1,raw,3,0.4335,0.4159,0.3512,0.2483
2,raw,5,0.4208,0.4284,0.2679,0.1517
3,raw,10,0.3908,0.4564,0.2083,0.0828
4,global,1,0.4673,0.4157,0.5327,0.4586
5,global,3,0.4325,0.4150,0.3571,0.2552
6,global,5,0.4196,0.4267,0.2708,0.1552
7,global,10,0.3902,0.4557,0.2113,0.0862
8,per_query,1,0.5298,0.4643,0.4702,0.3862
9,per_query,3,0.4702,0.4548,0.3452,0.2414



SENSITIVITY (label == 2):


,pipeline,K,P@K,Failure@K,Conditional_Failure@K
0,raw,1,0.1518,0.8482,0.6623
1,raw,3,0.1389,0.7381,0.4172
2,raw,5,0.1280,0.6905,0.3113
3,raw,10,0.1080,0.6190,0.1523
4,global,1,0.1518,0.8482,0.6623
5,global,3,0.1409,0.7440,0.4305
6,global,5,0.1268,0.6964,0.3245
7,global,10,0.1063,0.6280,0.1722
8,per_query,1,0.1815,0.8185,0.5960
9,per_query,3,0.1538,0.7173,0.3709


Global standardscaler is basically neutral here (tiny degradation). Nothing suggests a meaningful win for MQ2007.

Per-query normalization appears to push the model to pick a better #1, but it's less consistent at keeping at least one relevant doc inside the top-5 (on MQ2007). This can happen if the normalized features make the model more "peaky" at the very top but shuffle positions 2-5 in a worse way.


Even though per-query worsens Failure@5, it improves graded ranking quality (NDCG), meaning it might be placing higher-relevance items (label 2) earlier even if it occasionally "misses" relevance in top-5 for some queries.


Sensitivity:
It is mixed. Per-query helps a bit at K=5 (conditional) but not at K=10 (conditional) compared to raw.

**Overall** : 
Global scaling does not help MQ2007 (differences are tiny and slightly worse across multiple metrics).

Per-query normalization improves top-1 performance a lot (both P@1 and Failure@1 and conditional Failure@1).

Per-query normalization improves NDCG across K, meaning better graded ordering overall.

Per-query normalization hurts Failure@5 and conditional Failure@5, which is our “avoidable failure” lens.

## 8.2 Displaying MQ2008 Results

In [32]:
print("="*80)
print("MQ2008 AGGREGATE RESULTS")
print("="*80)

print("\nPRIMARY (label >= 1):")
display(agg_primary_2008[['pipeline', 'K', 'P@K', 'NDCG@K', 'Failure@K', 'Conditional_Failure@K']])

print("\nSENSITIVITY (label == 2):")
display(agg_sensitivity_2008[['pipeline', 'K', 'P@K', 'Failure@K', 'Conditional_Failure@K']])

MQ2008 AGGREGATE RESULTS

PRIMARY (label >= 1):


,pipeline,K,P@K,NDCG@K,Failure@K,Conditional_Failure@K
0,raw,1,0.4167,0.3526,0.5833,0.3810
1,raw,3,0.3825,0.4016,0.4359,0.1619
2,raw,5,0.3500,0.4460,0.3782,0.0762
3,raw,10,0.2643,0.4820,0.3397,0.0190
4,global,1,0.4295,0.3611,0.5705,0.3619
5,global,3,0.3825,0.4025,0.4359,0.1619
6,global,5,0.3513,0.4462,0.3782,0.0762
7,global,10,0.2649,0.4819,0.3397,0.0190
8,per_query,1,0.4359,0.3718,0.5641,0.3524
9,per_query,3,0.3889,0.4148,0.4295,0.1524



SENSITIVITY (label == 2):


,pipeline,K,P@K,Failure@K,Conditional_Failure@K
0,raw,1,0.2115,0.7885,0.4762
1,raw,3,0.1688,0.6795,0.2063
2,raw,5,0.1385,0.6282,0.0794
3,raw,10,0.0991,0.6154,0.0476
4,global,1,0.2179,0.7821,0.4603
5,global,3,0.1688,0.6795,0.2063
6,global,5,0.1359,0.6282,0.0794
7,global,10,0.0998,0.6154,0.0476
8,per_query,1,0.2115,0.7885,0.4762
9,per_query,3,0.1731,0.6667,0.1746


**Cross-dataset story (MQ2007 vs MQ2008)**

Global normalization:
Neutral in both datasets.
Sometimes tiny better at K=1, but overall it’s not the driver.

Per-query normalization
Consistent pattern across datasets in primary:
Improves top-1 a lot (strong in MQ2007, moderate in MQ2008)
Does not reduce conditional Failure@5 (hurts in both)
Improves NDCG (strong in MQ2007, small in MQ2008)

Sensitivity differs by dataset:
MQ2007: per-query not clearly better at K=10 conditional sensitivity
MQ2008: per-query is much better at K=10 conditional sensitivity

That’s real “generalization” evidence — not that it’s always best, but that it consistently trades off:
better early rank and graded ordering,
but potentially worse “at least one relevant in top-5” for primary.

# 9 Tie Diagnostics

In [36]:
print("="*80)
print("TIE DIAGNOSTICS")
print("="*80)

tie_rows=[]

for dataset_name, qm_dict in [('MQ2007', query_metrics_2007), ('MQ2008', query_metrics_2008)]:
    for pipeline_name, qm in qm_dict.items():
        for k in [5, 10]:
            tie_rows.append({
                'dataset': dataset_name,
                'pipeline': pipeline_name,
                'K': k,
                'mean_unique_scores': qm[f'unique_scores@{k}'].mean(),
                'mean_num_ties': qm[f'num_ties@{k}'].mean(),
                'pct_tie_at_cutoff': qm[f'tie_at_cutoff@{k}'].mean()*100
            })

tie_diagnostics=pd.DataFrame(tie_rows)

print("\nTie Diagnostics Summary:")
display(tie_diagnostics)

print("\nInterpretation:")
print("-unique_scores@K: Average number of distinct scores in top-K")
print("-num_ties@K: Average number of ties (K - unique_scores)")
print("-pct_tie_at_cutoff: Percentage of queries with tie at rank K")

TIE DIAGNOSTICS

Tie Diagnostics Summary:


,dataset,pipeline,K,mean_unique_scores,mean_num_ties,pct_tie_at_cutoff
0,MQ2007,raw,5,5.0000,0.0000,0.2976
1,MQ2007,raw,10,9.9940,0.0060,0.0000
2,MQ2007,global,5,5.0000,0.0000,0.2976
3,MQ2007,global,10,9.9940,0.0060,0.2976
4,MQ2007,per_query,5,5.0000,0.0000,0.0000
5,MQ2007,per_query,10,9.9970,0.0030,0.0000
6,MQ2008,raw,5,4.9936,0.0064,0.0000
7,MQ2008,raw,10,8.9103,0.0192,0.0000
8,MQ2008,global,5,4.9936,0.0064,0.0000
9,MQ2008,global,10,8.9103,0.0192,0.0000



Interpretation:
-unique_scores@K: Average number of distinct scores in top-K
-num_ties@K: Average number of ties (K - unique_scores)
-pct_tie_at_cutoff: Percentage of queries with tie at rank K
